# 🏸 BMCA - Badminton Match Coaching Assistant

Upload a badminton match video and get AI-powered coaching insights.

**Steps:**
1. Run Setup cell (installs dependencies)
2. Upload your video
3. Run the pipeline
4. Download the report and load it in the local UI

In [ ]:
#@title 1. Setup (Run once) { display-mode: "form" }
!pip install -q torch torchvision ultralytics onnxruntime opencv-python-headless scipy pyyaml gdown

# Clone repo if not already present
import os
if not os.path.exists('baddyCoach'):
    !git clone https://github.com/your-username/baddyCoach.git || echo 'Using local files'

# Download model weights
os.makedirs('baddyCoach/ckpts/rtmpose', exist_ok=True)
os.makedirs('baddyCoach/ckpts/bst', exist_ok=True)

print('✅ Setup complete! Proceed to Step 2.')

In [ ]:
#@title 2. Upload Video { display-mode: "form" }
from google.colab import files
import shutil

print('Upload your badminton match video (MP4, MOV, or AVI):')
uploaded = files.upload()

video_filename = list(uploaded.keys())[0]
video_path = f'/content/{video_filename}'
print(f'\n✅ Uploaded: {video_filename} ({os.path.getsize(video_path) / 1024 / 1024:.1f} MB)')
print(f'   Path: {video_path}')

In [ ]:
#@title 3. Run Pipeline { display-mode: "form" }
#@markdown This will process your video through 14 AI stages.
#@markdown On GPU (T4), expect ~5-15 minutes for a 30-minute video.

%cd /content/baddyCoach
!python colab/pipeline.py "{video_path}" --output report.json --device cuda

print('\n✅ Pipeline complete!')
print('   Report saved to: /content/baddyCoach/report.json')

In [ ]:
#@title 4. View Summary & Download Report { display-mode: "form" }
import json
from IPython.display import display, HTML

report_path = '/content/baddyCoach/report.json'
if not os.path.exists(report_path):
    print('❌ Report not found. Run Step 3 first.')
else:
    report = json.load(open(report_path))

    # Summary
    print('═' * 50)
    print('  MATCH ANALYSIS SUMMARY')
    print('═' * 50)
    print(f'  Total rallies: {len(report.get("rallies", []))}')
    print(f'  Total shots: {report.get("shot_count", 0)}')
    print(f'  Players: {len(report.get("footwork", {}))}')

    # Shot distribution
    sd = report.get("shot_distribution", {})
    if sd:
        print(f'\n  Shot Distribution:')
        for stroke, pct in sorted(sd.items(), key=lambda x: -x[1])[:5]:
            print(f'    {stroke:15s} {pct*100:5.1f}%')

    # Coach recommendations
    strengths = report.get("strengths", [])
    weaknesses = report.get("weaknesses", [])
    if strengths:
        print(f'\n  ✅ Strengths:')
        for s in strengths[:3]:
            print(f'    • {s[:80]}')
    if weaknesses:
        print(f'\n  ⚠️  Areas to improve:')
        for w in weaknesses[:3]:
            print(f'    • {w[:80]}')

    print('\n' + '═' * 50)

    # Download
    from google.colab import files
    files.download(report_path)
    print('\n📥 Report downloaded! Load it in the local BMCA UI.')